# SupplyMind — Exploratory Analysis

Quick tour of the synthetic supplier network, critical-node analysis, demand forecasting, and the causal effect estimate. Run from the project root with the `src/` package importable (e.g. `pip install -e .` or run Jupyter from the repo root).

In [ ]:
import sys
sys.path.insert(0, '..')

from src.graph.build_graph import build_and_save_default_graph
from src.graph.graph_utils import identify_critical_nodes, simulate_disruption_propagation

g = build_and_save_default_graph()
print(f'{g.number_of_nodes()} nodes, {g.number_of_edges()} edges')

## Critical failure points

Composite score = betweenness centrality + baseline risk + inverse reliability.

In [ ]:
ranked = identify_critical_nodes(g, top_k=10)
for node_id, score in ranked:
    print(f'{node_id:>10s}  {score:.3f}')

## Disruption propagation simulation

BFS cascade from the top critical node, following edges whose dependency weight exceeds the propagation threshold.

In [ ]:
origin = ranked[0][0]
result = simulate_disruption_propagation(g, origin)
print(f'Disrupting {origin} affects {len(result.affected_nodes)} downstream nodes over {len(result.propagation_steps)} hops')
print(f'Estimated capacity lost: {result.total_capacity_lost:,.0f} units')
result.affected_nodes

## Demand forecast (Prophet)

In [ ]:
from src.forecasting.data_prep import load_demand_data
from src.forecasting.prophet_model import ProphetForecaster

df = load_demand_data()
entity_df = df[df.entity_id == 'MFG_000'][['ds', 'y']]
forecaster = ProphetForecaster().fit(entity_df)
forecast = forecaster.predict(periods=30)
forecast.tail()

## Causal effect of disruption on downstream impact (DoWhy)

In [ ]:
from src.causal.causal_model import run_causal_analysis

result = run_causal_analysis(g)
print(result.summary)
print(f'Placebo refutation passed: {result.placebo_refutation_passed}')
print(f'Random common cause refutation passed: {result.random_common_cause_refutation_passed}')